In [1]:
import os

print("--- Scanning /kaggle/working ---")
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if "summary" in file.lower() or "output" in file.lower() or file.endswith(('.csv', '.txt')):
            print(f"Found file: {os.path.join(root, file)}")

print("\n--- Listing directory contents directly ---")
if os.path.exists("/kaggle/working/veritas_mistral7b_final/output"):
    print("Contents of target folder:", os.listdir("/kaggle/working/veritas_mistral7b_final/output"))
else:
    print("Target folder does not exist! Checking parents...")
    if os.path.exists("/kaggle/working/veritas_mistral7b_final"):
        print("Contents of veritas_mistral7b_final:", os.listdir("/kaggle/working/veritas_mistral7b_final"))

--- Scanning /kaggle/working ---

--- Listing directory contents directly ---
Target folder does not exist! Checking parents...


In [14]:
import os

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))

/kaggle/input/datasets/rithwikasai/test-json/test.json


In [1]:
"""
veritas_mistral7b_final.py
==========================
Mistral-7B-Instruct + VERITAS pipeline with POSITIONAL ROLE PARTITIONING.
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ─── HUGGING FACE AUTHENTICATION ─────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("[INIT] Successfully loaded Hugging Face token from Kaggle Secrets.")
except Exception as e:
    print(f"[WARNING] Kaggle Secrets unavailable: {e}")
    if not os.environ.get("HF_TOKEN"):
        raise EnvironmentError(
            "[INIT] HF_TOKEN not found in Kaggle Secrets or environment. "
            "Please add it via Kaggle Notebook Secrets before running."
        )
    print("[INIT] Using HF_TOKEN already present in environment.")

import re, csv, json, gc, math, statistics as st
import numpy as np
import torch
import sys as _sys

# ─── CONFIG ──────────────────────────────────────────────────────────────────
SOURCE_IDS = "all"
_cli = [a for a in _sys.argv[1:] if a.strip() and
        (a.strip().lower() == "all" or a.strip().isdigit())]
_env = os.environ.get("VERITAS_IDS", "").strip()
if _cli:
    if any(a.lower() == "all" for a in _cli):
        SOURCE_IDS = "all"
    else:
        SOURCE_IDS = _cli
elif _env:
    SOURCE_IDS = "all" if _env.lower() == "all" else [x for x in re.split(r"[,\s]+", _env) if x]

GEN_MODEL_ID   = "mistralai/Mistral-7B-Instruct-v0.3"
MAX_NEW_TOKENS = 150
N_CHUNKS       = 7
EMBED_MODEL    = "all-MiniLM-L6-v2"
NLI_MODEL      = "cross-encoder/nli-deberta-v3-small"
TOP_K          = 6
COV_THRESH     = 0.60
OMIT_K         = 4
FAITH_OK       = 0.50   # Base threshold; per-position thresholds override below
STRONG_MODE    = True
STRICT         = True
OUT_DIR        = "/kaggle/working/veritas_mistral7b_final/output"
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# ─── POSITIONAL ROLE CONFIG (First < Middle < Last severity) ─────────────────
POSITION_CONFIG = {
    "First":  {"top_k": 2, "temperature": 0.3, "nli_check": False, "nli_threshold": 0.0,  "max_retries": 0},
    "Middle": {"top_k": 3, "temperature": 0.2, "nli_check": True,  "nli_threshold": 0.40, "max_retries": 2},
    "Last":   {"top_k": 4, "temperature": 0.1, "nli_check": True,  "nli_threshold": 0.50, "max_retries": 3},
}

POSITION_INSTRUCTION = {
    "First":  ("You are correcting the OPENING sentence of a legal-case summary. "
               "Lightly rewrite it to remove unsupported claims. Use ONLY the context."),
    "Middle": ("You are correcting a MIDDLE sentence of a legal-case summary. "
               "It often has fabricated dates/sections/names. Rewrite strictly using ONLY "
               "the context; drop unsupported claims."),
    "Last":   ("You are correcting the FINAL sentence of a legal-case summary. "
               "These have the highest hallucination. Use ONLY what is explicitly in the "
               "context; remove any unverifiable claim. One sentence only."),
}


# ─── AUTOMATIC DATASET PATH DISCOVERY ────────────────────────────────────────
def resolve_src_path():
    PREFERRED_NAMES = {"test.json", "test_data.json", "test-data.json"}
    fallbacks = [
        "/kaggle/input/datasets/rithwikasai/test-json/test.json",
        "/kaggle/input/datasets/alphamom/test-data",
        "/kaggle/input/datasets/alphamom/test-data/test_data.json",
        "/kaggle/input/datasets/rithwikasai.test_data.json/test_data.json",
        "/kaggle/input/test_data.json",
        "/kaggle/working/test_data.json",
    ]
    for p in fallbacks:
        if os.path.isfile(p): return p
        if os.path.isdir(p):
            for f in os.listdir(p):
                if f.endswith(".json"): return os.path.join(p, f)

    print("[INIT] Scanning /kaggle/input for json datasets...")
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f.lower() in PREFERRED_NAMES:
                    resolved = os.path.join(root, f)
                    print(f"[INIT] Located: {resolved}")
                    return resolved
        for root, _, files in os.walk("/kaggle/input"):
            for f in files:
                if f.endswith(".json"):
                    resolved = os.path.join(root, f)
                    print(f"[INIT] Fallback .json: {resolved}")
                    return resolved

    raise FileNotFoundError("Could not discover data file in environment inputs.")

SRC_PATH = resolve_src_path()


# ─── LOGIC HELPERS ───────────────────────────────────────────────────────────
def load_records(path):
    with open(path, "r", encoding="utf-8") as f: d = json.load(f)
    return d if isinstance(d, list) else [d]

def split_sentences(t):
    t = " ".join(str(t).split())
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', t) if len(s.strip()) > 15]

_STOP = set("the and that this with from were have has had was are for not but which their "
            "they them his her she him who been into under upon such shall would could "
            "there these those than then also".split())
def _kw(t): return {w for w in re.findall(r'[A-Za-z]{4,}', t.lower()) if w not in _STOP}

_SEC_RE = re.compile(r'(?:section|article|s\.|art\.)\s*\d+', re.I)
_NUM_RE = re.compile(r'\b\d[\d,\./-]*\b')

try:
    import spacy
    try:    _NLP = spacy.load("en_core_web_sm")
    except Exception: _NLP = None
except Exception: _NLP = None

def entities(t):
    if _NLP is not None:
        return [e.text.strip() for e in _NLP(t).ents
                if e.label_ in {"PERSON","ORG","GPE","LOC","DATE","LAW","NORP","FAC"}]
    return re.findall(r'\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', t)

def atoms_of(s):
    secs = [m.group(0) for m in _SEC_RE.finditer(s)]
    nums = [m.group(0) for m in _NUM_RE.finditer(s)
            if not any(m.group(0) in x for x in secs)]
    return secs + nums + entities(s)

ROLE_W = {"Ratio":1.0,"Relief":0.8,"Arguments":0.6,"Facts":0.4,"Obiter":0.2}
def role_of(s):
    sl = s.lower()
    if re.search(r'\b(held|ruled|found that|concluded|reasoned)\b', sl):          return "Ratio"
    if re.search(r'\b(argued|contended|submitted|appellant|respondent)\b', sl):   return "Arguments"
    if re.search(r'\b(dismissed|allowed|granted|set aside|decree|appeal)\b', sl): return "Relief"
    if re.search(r'\b(observed|noted|obiter)\b', sl):                             return "Obiter"
    return "Facts"

def supported(atom, src):
    return atom.replace(",","").strip().lower() in src or atom.lower() in src

def clean_output(text, original):
    text = text.strip().strip('"').strip()
    m = re.search(r'(corrected sentence|fixed sentence|would be)\s*[:\-]\s*', text, re.I)
    if m: text = text[m.end():].strip()
    for pre in ("Corrected sentence:", "Corrected:", "Answer:", "Fixed:", "Summary:"):
        if text.lower().startswith(pre.lower()): text = text[len(pre):].strip()
    text = re.sub(r'\s{2,}', ' ', text).strip().strip('-').strip()
    if not text or text.lower() == "none": return original
    if not text.endswith((".", "!", "?")): text += "."
    return text


# ─── PHASE 1: MISTRAL GENERATION ENGINE ──────────────────────────────────────
def run_generation_phase(records, ids_to_process):
    print(f"\n--- [PHASE 1] Initializing {GEN_MODEL_ID} (FP16) ---")
    from transformers import AutoTokenizer, AutoModelForCausalLM

    _gen_tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
    _gen_tok.padding_side = "left"
    if _gen_tok.pad_token is None:
        _gen_tok.pad_token = _gen_tok.eos_token

    if DEVICE == "cuda":
        _gen_model = AutoModelForCausalLM.from_pretrained(
            GEN_MODEL_ID, torch_dtype=torch.float16,
            device_map="auto", attn_implementation="sdpa"
        )
    else:
        _gen_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_ID, torch_dtype=torch.float32)
    _gen_model.eval()

    _INSTRUCTION = """You are an expert legal clerk. Extract a highly accurate, structured summary of the provided court judgment.
STRICT RULES: DO NOT invent dates, sections, or names. Use ONLY information explicitly stated.
REQUIRED STRUCTURE:
- BACKGROUND: (Core dispute)
- RATIO DECIDENDI: (Core legal reasoning)
- FINAL DECISION: (Exact relief or appeal status)"""

    def _mistral_one(chunk_text):
        prompt = f"<s>[INST] {_INSTRUCTION}\n\n<judgment>\n{chunk_text[:1800]}\n</judgment>\n\nSummary: [/INST]"
        inp = _gen_tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
        inp = {k: v.to(_gen_model.device) for k, v in inp.items()}
        with torch.no_grad():
            ids = _gen_model.generate(
                **inp, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                repetition_penalty=1.15, pad_token_id=_gen_tok.eos_token_id, use_cache=True
            )
        out = _gen_tok.decode(ids[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        out = " ".join(out.replace("\n", " ").split())
        for prefix in ["Summary:", "Answer:"]:
            if out.lower().startswith(prefix.lower()): out = out[len(prefix):].strip()
        return out

    generated_outputs = {}
    by_id = {str(r.get("id")): r for r in records}
    for sid in ids_to_process:
        rec = by_id.get(sid)
        if rec is None: continue
        judgment = rec.get("judgment", "")
        js     = split_sentences(judgment) or [judgment]
        n      = max(1, min(N_CHUNKS, len(js)))
        size   = max(1, len(js) // n)
        chunks = [" ".join(js[i:i+size]) for i in range(0, len(js), size)][:N_CHUNKS]
        parts  = []
        for ch in chunks:
            try:
                parts.append(_mistral_one(ch))
            except Exception as e:
                if STRICT: raise e
                parts.append(ch.split(".")[0].strip() + ".")
        generated_outputs[sid] = " ".join(p for p in parts if p.strip())
        print(f"   [PHASE 1] Generated: Source ID {sid}")

    # Save raw generated summaries
    os.makedirs(OUT_DIR, exist_ok=True)
    raw_csv_path = os.path.join(OUT_DIR, "generated_summaries_raw.csv")
    with open(raw_csv_path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["source_id", "generated_summary"])
        w.writeheader()
        w.writerows([{"source_id": sid, "generated_summary": txt}
                     for sid, txt in generated_outputs.items()])
    print(f"   [PHASE 1] Raw summaries saved to: {raw_csv_path}")

    del _gen_model, _gen_tok
    if DEVICE == "cuda": torch.cuda.empty_cache()
    gc.collect()
    print("--- [PHASE 1] Mistral flushed from VRAM ---")
    return generated_outputs


# ─── RUN PIPELINE ────────────────────────────────────────────────────────────
os.makedirs(OUT_DIR, exist_ok=True)
records = load_records(SRC_PATH)
by_id   = {str(r.get("id")): r for r in records}
ids     = list(by_id.keys()) if SOURCE_IDS == "all" else [str(i) for i in SOURCE_IDS]

print(f"[RUN] Dataset: {SRC_PATH}")
print(f"[RUN] {len(records)} judgments | Processing: {ids}")

generated_summaries = run_generation_phase(records, ids)

print(f"\n--- [PHASE 2] Loading Embedder + NLI + Mistral for positional mitigation ---")
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM

emb = SentenceTransformer(EMBED_MODEL, device=DEVICE)
nli = CrossEncoder(NLI_MODEL, device=DEVICE)
label2id = nli.model.config.label2id
ENTAILMENT_IDX = label2id.get("ENTAILMENT", label2id.get("entailment", 2))
print(f"[PHASE 2] NLI entailment index: {ENTAILMENT_IDX}")

# Reload Mistral for positional sentence-level correction
print(f"[PHASE 2] Reloading {GEN_MODEL_ID} for positional correction...")
_cor_tok = AutoTokenizer.from_pretrained(GEN_MODEL_ID)
_cor_tok.padding_side = "left"
if _cor_tok.pad_token is None: _cor_tok.pad_token = _cor_tok.eos_token

if DEVICE == "cuda":
    _cor_model = AutoModelForCausalLM.from_pretrained(
        GEN_MODEL_ID, torch_dtype=torch.float16,
        device_map="auto", attn_implementation="sdpa"
    )
else:
    _cor_model = AutoModelForCausalLM.from_pretrained(GEN_MODEL_ID, torch_dtype=torch.float32)
_cor_model.eval()
print("[PHASE 2] All components loaded.")


# ─── VERITAS HELPERS ─────────────────────────────────────────────────────────
_mem = {}
def memory(rec):
    sid = rec.get("id")
    if sid not in _mem:
        js = split_sentences(rec.get("judgment","")) or [rec.get("judgment","")]
        je = emb.encode(js, convert_to_numpy=True, normalize_embeddings=True)
        _mem[sid] = {"sents": js, "emb": je, "src": rec.get("judgment","").lower()}
    return _mem[sid]

def retrieve(query, rec, k=TOP_K):
    m = memory(rec)
    q = emb.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    return [m["sents"][i] for i in (m["emb"] @ q).argsort()[::-1][:k]]

def faith_score(ev_list, hyp):
    if not hyp.strip() or not ev_list: return 0.0
    scores = nli.predict([(e, hyp) for e in ev_list], apply_softmax=True)
    return round(float(max(s[ENTAILMENT_IDX] for s in scores)), 4)

def entail_single(premise, hyp):
    if not hyp.strip(): return 0.0
    return float(nli.predict([(premise, hyp)], apply_softmax=True)[0][ENTAILMENT_IDX])


def positional_correct(sentence, position, context_sents):
    """Rewrite sentence using Mistral with position-aware strictness."""
    cfg         = POSITION_CONFIG[position]
    instruction = POSITION_INSTRUCTION[position]
    context     = " ".join(context_sents[:cfg["top_k"]])

    prompt = (f"<s>[INST] {instruction}\n\n"
              f"Context:\n{context[:1500]}\n\n"
              f"Original sentence: \"{sentence}\"\n\n"
              f"Corrected sentence: [/INST]")

    inp = _cor_tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
    inp = {k: v.to(_cor_model.device) for k, v in inp.items()}

    with torch.no_grad():
        out_ids = _cor_model.generate(
            **inp,
            max_new_tokens      = 80,
            do_sample           = cfg["temperature"] > 0,
            temperature         = cfg["temperature"] if cfg["temperature"] > 0 else 1.0,
            top_k               = cfg["top_k"],
            repetition_penalty  = 1.15,
            pad_token_id        = _cor_tok.eos_token_id,
            use_cache           = True
        )
    raw = _cor_tok.decode(out_ids[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return clean_output(raw, sentence)


def fix_with_position(sentence, position, rec):
    """
    Position-aware hallucination fix:
    - First:  light atom removal only, no NLI retry
    - Middle: atom removal + NLI check >= 0.40, up to 2 retries
    - Last:   atom removal + positional correction + NLI >= 0.50, up to 3 retries
    """
    src = memory(rec)["src"]
    ev  = retrieve(sentence, rec, k=POSITION_CONFIG[position]["top_k"] + 2)
    if not ev:
        return sentence, 0.0, 0.0

    cfg   = POSITION_CONFIG[position]
    f0    = faith_score(ev, sentence)

    # Step 1: Atom-level surgical excision (all positions)
    healed = sentence
    for a in sorted([x for x in atoms_of(sentence) if not supported(x, src)], key=len, reverse=True):
        healed = re.sub(r'\s*' + re.escape(a) + r'\s*', ' ', healed, count=1)
    healed = re.sub(r'\s+([,.;])', r'\1', healed)
    healed = re.sub(r'\s{2,}', ' ', healed).strip(" ,;")
    if healed and not healed.endswith((".", "!", "?")): healed += "."
    if not healed.strip():
        print(f"[WARN] Atom stripping emptied: '{sentence[:60]}...'")
        healed = sentence

    healed_score = faith_score(ev, healed)

    # Step 2: Pick best of original vs healed
    text, fbest = (healed, healed_score) if healed_score >= f0 else (sentence, f0)

    # Step 3: NLI-gated positional correction with retries (Middle + Last)
    if cfg["nli_check"]:
        nli_thresh = cfg["nli_threshold"]
        for attempt in range(cfg["max_retries"] + 1):
            if fbest >= nli_thresh:
                break
            corrected = positional_correct(sentence, position, ev)
            c_score   = faith_score(ev, corrected)
            if c_score > fbest:
                text, fbest = corrected, c_score
            if attempt < cfg["max_retries"]:
                print(f"   [{position}] retry {attempt+1}/{cfg['max_retries']} "
                      f"faith={fbest:.3f} (need {nli_thresh})")

    # Step 4: Monotonic guard — never go below original
    if fbest < f0:
        text, fbest = sentence, f0

    # Step 5: Strong mode fallback — use best evidence if still weak
    if STRONG_MODE and fbest < FAITH_OK:
        best_src, best_sf = max(((e, faith_score(ev, e)) for e in ev), key=lambda c: c[1])
        if best_sf > fbest:
            text, fbest = best_src, best_sf

    return text, fbest, f0


def importance(sent, summ_kw):
    sk   = _kw(sent)
    ov   = len(sk & summ_kw) / (len(sk) + 1)
    dens = len(atoms_of(sent)) / (len(sent.split()) + 1)
    return 0.4*ov + 0.3*min(dens*5, 1.0) + 0.3*ROLE_W.get(role_of(sent), 0.4)

def omission_rank(sent, summ_kw):
    return (importance(sent, summ_kw), ROLE_W.get(role_of(sent), 0.4),
            len(atoms_of(sent)), len(_kw(sent) & summ_kw))

def coverage_omissions(summary_sents, rec):
    js = memory(rec)["sents"]
    if not summary_sents: return 0.0, js[:OMIT_K]
    je      = memory(rec)["emb"]
    se      = emb.encode(summary_sents, convert_to_numpy=True, normalize_embeddings=True)
    maxsim  = (je @ se.T).max(axis=1)
    cov     = round(float((maxsim >= COV_THRESH).mean()), 4)
    summ_kw = set().union(*[_kw(s) for s in summary_sents])
    omit    = sorted([i for i in range(len(js)) if maxsim[i] < COV_THRESH],
                     key=lambda i: omission_rank(js[i], summ_kw), reverse=True)[:OMIT_K]
    return cov, [js[i] for i in omit]


def mitigate_and_analyze(sid, gen_text, rec):
    gen_sents = split_sentences(gen_text) or [gen_text]
    num_bins  = 5
    bin_size  = math.ceil(len(gen_sents) / num_bins) if gen_sents else 1

    positions, hall_scores = [], []
    faithful, fb_scores, fixed_n = [], [], 0

    for i, s in enumerate(gen_sents):
        bin_id = min((i // bin_size) + 1, num_bins) if bin_size > 0 else 1
        if bin_id <= 2:   pos = "First"
        elif bin_id == 3: pos = "Middle"
        else:             pos = "Last"
        positions.append(pos)

        # Use position-aware fix
        ftext, ff, f0 = fix_with_position(s, pos, rec)
        fb_scores.append(f0)
        if ftext.strip().lower() != s.strip().lower(): fixed_n += 1
        faithful.append(ftext)
        hall_scores.append(1.0 - ff)

    cov_before, omitted = coverage_omissions(faithful, rec)
    completed           = faithful + omitted
    cov_after, _        = coverage_omissions(completed, rec)

    faith_before = round(st.mean(fb_scores), 4) if fb_scores else 0.0
    fa_scores    = [faith_score(retrieve(s, rec), s) for s in completed]
    faith_after  = round(st.mean(fa_scores), 4)  if fa_scores else 0.0

    overall_hall_pct = round((sum(hall_scores) / len(hall_scores)) * 100, 2) if hall_scores else 0.0

    pos_scores = {"First": [], "Middle": [], "Last": []}
    for p, score in zip(positions, hall_scores):
        pos_scores[p].append(score)
    trends = {p: round((sum(pos_scores[p]) / len(pos_scores[p])) * 100, 2) if pos_scores[p] else 0.0
              for p in ["First", "Middle", "Last"]}

    return " ".join(completed), faith_before, faith_after, cov_before, cov_after, fixed_n, len(omitted), overall_hall_pct, trends, gen_sents


# ─── EVALUATION LOOP ─────────────────────────────────────────────────────────
summaries, report = [], []
skipped = 0

print(f"\n--- [PHASE 2] Positional Mitigation Sequence ---")
for sid in ids:
    rec = by_id.get(sid)
    gen = generated_summaries.get(sid, "")
    if not gen.strip():
        print(f"[ERROR] source_id {sid}: blank output. Skipping.")
        skipped += 1
        continue

    final_text, faith_before, faith_after, cov_before, cov_after, fixed_n, omit_n, overall_hall, trends, gen_sents = \
        mitigate_and_analyze(sid, gen, rec)

    print(f"=== source_id {sid} === Sentences: {len(gen_sents)}")
    print(f"   faith {faith_before}->{faith_after} | coverage {cov_before}->{cov_after} "
          f"| fixed={fixed_n} | omissions_added={omit_n}")
    print(f"   Hall%: {overall_hall}% | First: {trends['First']}% | "
          f"Mid: {trends['Middle']}% | Last: {trends['Last']}%\n")

    summaries.append({
        "source_id":                 sid,
        "n_gen_sentences":           len(gen_sents),
        "generated_summary":         gen,
        "final_mitigated_summary":   final_text,
        "overall_hallucination_pct": overall_hall,
        "trend_first":               trends["First"],
        "trend_middle":              trends["Middle"],
        "trend_last":                trends["Last"]
    })
    report.append({
        "source_id":            sid,
        "n_gen_sentences":      len(gen_sents),
        "faith_before":         faith_before,
        "faith_after":          faith_after,
        "coverage_before":      cov_before,
        "coverage_after":       cov_after,
        "hallucinations_fixed": fixed_n,
        "omissions_added":      omit_n,
        "overall_hall_pct":     overall_hall
    })

# Flush correction model
del _cor_model, _cor_tok
if DEVICE == "cuda": torch.cuda.empty_cache()
gc.collect()

# ─── PERSIST OUTPUT ARTIFACTS ────────────────────────────────────────────────
with open(os.path.join(OUT_DIR, "veritas_mistral7b_summaries.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["source_id","n_gen_sentences","generated_summary",
                                       "final_mitigated_summary","overall_hallucination_pct",
                                       "trend_first","trend_middle","trend_last"])
    w.writeheader(); w.writerows(summaries)

with open(os.path.join(OUT_DIR, "veritas_mistral7b_report.csv"), "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["source_id","n_gen_sentences","faith_before","faith_after",
                                       "coverage_before","coverage_after","hallucinations_fixed",
                                       "omissions_added","overall_hall_pct"])
    w.writeheader(); w.writerows(report)

if report:
    fb = [r["faith_before"]    for r in report]
    fa = [r["faith_after"]     for r in report]
    cb = [r["coverage_before"] for r in report]
    ca = [r["coverage_after"]  for r in report]
    skipped_line = f"judgments skipped (empty)   : {skipped}" + (
        "  ← WARNING: some records produced no output" if skipped > 0 else "")
    cert = (
        "VERITAS-MISTRAL7B CERTIFICATE (Positional Role Partitioning)\n"
        "====================================================================\n"
        "judgments processed         : {n}\n"
        "{skipped_line}\n"
        "generator                   : {gm}\n"
        "positional mitigation       : First(top_k=2) < Middle(top_k=3,NLI>=0.40) < Last(top_k=4,NLI>=0.50)\n"
        "avg FAITHFULNESS            : {fb} -> {fa}   (hallucination mitigated)\n"
        "avg COVERAGE                : {cb} -> {ca}   (omission handled)\n"
        "total hallucinations fixed  : {hf}\n"
        "total omissions added       : {oa}\n"
    ).format(
        n=len(report), skipped_line=skipped_line, gm=GEN_MODEL_ID,
        fb=round(st.mean(fb), 4), fa=round(st.mean(fa), 4),
        cb=round(st.mean(cb), 4), ca=round(st.mean(ca), 4),
        hf=sum(r["hallucinations_fixed"] for r in report),
        oa=sum(r["omissions_added"]      for r in report)
    )
    with open(os.path.join(OUT_DIR, "veritas_mistral7b_certificate.txt"), "w", encoding="utf-8") as f:
        f.write(cert)
    print("="*60 + "\n" + cert.strip() + "\n" + "="*60)

print(f"Pipeline Completed. Outputs saved to: {OUT_DIR}\nDONE")

[INIT] Successfully loaded Hugging Face token from Kaggle Secrets.
[RUN] Dataset: /kaggle/input/datasets/rithwikasai/test-json/test.json
[RUN] 100 judgments | Processing: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99']

--- [PHASE 1] Initializing mistralai/Mistral-7B-Instruct-v0.3 (FP16) ---


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

[PHASE 2] NLI entailment index: 1
[PHASE 2] Reloading mistralai/Mistral-7B-Instruct-v0.3 for positional correction...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

[PHASE 2] All components loaded.

--- [PHASE 2] Positional Mitigation Sequence ---
   [Middle] retry 1/2 faith=0.005 (need 0.4)
   [Middle] retry 2/2 faith=0.013 (need 0.4)
   [Middle] retry 1/2 faith=0.009 (need 0.4)
   [Middle] retry 2/2 faith=0.009 (need 0.4)
   [Middle] retry 1/2 faith=0.987 (need 0.4)
   [Last] retry 1/3 faith=0.017 (need 0.5)
   [Last] retry 2/3 faith=0.017 (need 0.5)
   [Last] retry 3/3 faith=0.017 (need 0.5)
   [Last] retry 1/3 faith=0.030 (need 0.5)
   [Last] retry 2/3 faith=0.030 (need 0.5)
   [Last] retry 3/3 faith=0.030 (need 0.5)
   [Last] retry 1/3 faith=0.024 (need 0.5)
   [Last] retry 2/3 faith=0.075 (need 0.5)
   [Last] retry 3/3 faith=0.098 (need 0.5)
   [Last] retry 1/3 faith=0.003 (need 0.5)
   [Last] retry 2/3 faith=0.006 (need 0.5)
   [Last] retry 3/3 faith=0.006 (need 0.5)
=== source_id 0 === Sentences: 23
   faith 0.3856->0.965 | coverage 0.2984->0.3548 | fixed=14 | omissions_added=4
   Hall%: 3.82% | First: 2.32% | Mid: 1.28% | Last: 7.28%

   